# Notebook 05 — Direct S2TT D0 (Bahnar Speech → Vietnamese Text)

**RQ1 role:** train the Direct baseline using the same matched supervised speech examples as the Cascaded baseline.

Locked architecture: **XLS-R 300M speech encoder + mBART-50 Vietnamese decoder**. The model maps audio directly to Vietnamese text; `text_bahnar` is never a model input.

Scientific rules:
- Uses the **exact NB03 ASR-eligible train/validation UIDs** and joins only `text_vi` from locked RQ1 manifests.
- Initializes **only from public pretrained XLS-R + mBART-50**; never from NB03 fine-tuned ASR or NB04 fine-tuned MT checkpoints.
- Never opens `rq1_test.csv` in Notebook 05. Frozen test is reserved for Notebook 06 C0-vs-D0 evaluation.
- Stages: `prepare → pilot → resume_test_a → restart kernel → resume_test_b → train → evaluate`.
- Operational knobs come from `configs/direct.yaml`; scientific pins are fail-closed against `src.direct_contract`.
- `evaluate` restores the best durable D0 checkpoint via LATEST and scores full `text_vi_norm` references.


In [ ]:
# Cell 1 — RunPod environment (locked; assignment overrides inherited env)
import os
from pathlib import Path

os.environ["BAHNAR_LOCAL_ROOT"] = "/tmp/bahnar-runtime"
os.environ["BAHNAR_DURABLE_ROOT"] = "/workspace/bahnar-s2tt-thesis"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# Checkpoint budget is required at resume_test/train write stages only; no notebook default.
Path(os.environ["BAHNAR_LOCAL_ROOT"]).mkdir(parents=True, exist_ok=True)
for k in ("BAHNAR_LOCAL_ROOT", "BAHNAR_DURABLE_ROOT", "BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES", "HF_HUB_ENABLE_HF_TRANSFER"):
    print(k, "=", os.environ.get(k))


In [ ]:
# Cell 2 — Bootstrap
import sys
from pathlib import Path
_NB_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
_PROJECT_ROOT = _NB_DIR.parent if _NB_DIR.name == "notebooks" else _NB_DIR
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
print("Project root:", _PROJECT_ROOT)


In [ ]:
# Cell 3 — Imports
import json, math, os, platform, shutil, time, traceback, uuid
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import torch

from src.seed import set_seed
from src.asr_full_data import FULL_PREPARE_MAX_DURATION
from src.asr_full_train import (
    assert_checkpoint_complete_for_resume, assert_cross_session_resume, assert_no_frozen_test_access,
    durable_experiment_dir, ensure_experiment_fingerprint, make_durable_checkpoint_sync_callback,
    make_resume_proof_callback, make_uid_tracking_collator, make_uid_tracking_trainer_cls, new_session_token,
    resolve_resume_checkpoint, restore_experiment_checkpoints_from_durable,
    summarize_resume_proof, derive_resume_test_status_from_proof, sync_experiment_checkpoints_to_durable,
    write_checkpoint_fingerprint, list_step_checkpoints,
)
from src.direct_runtime_paths import FULL_TRAIN_MARKER, PILOT_MARKER, RESUME_TEST_MARKER, resolve_direct_runtime_paths
from src.direct_contract import (
    DIRECT_CONFIG_RELPATH, LOCKED_ASR_PREPARE_CONTRACT_HASH, LOCKED_DECODER_ID, LOCKED_DECODER_REVISION,
    LOCKED_DIRECT_MONITOR_SIZE, LOCKED_ENCODER_ID, LOCKED_ENCODER_REVISION, LOCKED_EXPERIMENT_ID,
    LOCKED_GREATER_IS_BETTER, LOCKED_HARD_MAX_TRUNCATION_RATE, LOCKED_HARD_MAX_UNK_RATE,
    LOCKED_MAX_AUDIO_DURATION, LOCKED_MAX_TARGET_LENGTH, LOCKED_METRIC_FOR_BEST_MODEL, LOCKED_SAMPLE_RATE,
    LOCKED_TARGET_LANG, MATCHED_INPUT_POLICY, STATUS_DIRECT_EVALUATE, STATUS_DIRECT_PILOT, STATUS_DIRECT_PREPARE,
    STATUS_DIRECT_RESUME_TEST, STATUS_DIRECT_TRAINING, STATUS_FAILED, TARGET_NORMALIZATION_VERSION,
    assert_direct_data_contract_self_consistent, assert_direct_training_contract_self_consistent,
    assert_direct_training_contracts_match, assert_locked_direct_baseline, assert_locked_direct_runtime,
    assert_source_fingerprint_matches, build_direct_data_contract, build_direct_generation_config,
    build_direct_training_contract, compute_source_fingerprint, load_direct_yaml_config,
)
from src.direct_data import (
    DIRECT_PREPARE_SUMMARY, DirectAccessTracker, assert_direct_frames_match_contract, frame_csv_sha256,
    hydrate_direct_audio, load_asr_matched_direct_frames, load_direct_prepare_success, load_direct_prepared_frames,
    load_latest_prepare_pointer, persist_direct_prepare, preflight_direct_local_disk,
)
from src.direct_model import audit_direct_targets_by_split, load_direct_model, load_direct_processors, parameter_report
from src.direct_dataset import DirectDataCollator, DirectSpeechTranslationDataset
from src.data_utils import sha256_file
from src.direct_full_train import (
    RESUME_TEST_PHASE_A_STEPS, RESUME_TEST_PHASE_B_STEPS, RESUME_TEST_SUBSET_SIZE,
    assert_direct_checkpoint_budget, build_direct_training_hparams, compute_direct_seq2seq_metrics,
    derive_direct_handoff, derive_direct_training_status, evaluate_prediction_rows, fixed_subset,
    assert_monitor_file_sha256, assert_monitor_matches_validation, frozen_flag_from_tracker,
    generate_direct_predictions, load_stage_success, make_resume_safe_seq2seq_trainer_cls, monitor_manifest, plan_direct_resume_position,
    restore_direct_best_checkpoint, validate_direct_best_checkpoint, write_json,
)
from src.direct_export import export_notebook05_run
from src.mt_tokenize import tokenizer_fingerprint
print("imports ok")


In [ ]:
# Cell 4 — Locked experiment configuration from configs/direct.yaml
RUN_MODE = "full"
FULL_STAGE = "prepare"  # prepare|pilot|resume_test_a|resume_test_b|train|evaluate
ALLOW_FULL_TRAINING = False  # set True only at FULL_STAGE=train

CFG = load_direct_yaml_config(_PROJECT_ROOT / DIRECT_CONFIG_RELPATH)
ENCODER_ID = CFG["encoder_id"]
ENCODER_REVISION = CFG["encoder_revision"]
DECODER_ID = CFG["decoder_id"]
DECODER_REVISION = CFG["decoder_revision"]
TARGET_LANG = CFG["target_lang"]
EXPERIMENT_ID = CFG["experiment_id"]
assert_locked_direct_baseline(ENCODER_ID, ENCODER_REVISION, DECODER_ID, DECODER_REVISION, TARGET_LANG)

DATASET_ID = "cuong06/Bahnar_Vietnamese"
EXPECTED_DATASET_REVISION = "3d88d3951b1a6e3388559b341cd7bd274879d696"
EXPECTED_PARQUET_REVISION = "ad0a84362053dad098b86d0df215eb081f891dd8"
EXPECTED_ASR_TRAIN_COUNT = CFG["nb03_train_count"]
EXPECTED_ASR_VALIDATION_COUNT = CFG["nb03_validation_count"]
SAMPLE_RATE = LOCKED_SAMPLE_RATE
MAX_AUDIO_DURATION = float(LOCKED_MAX_AUDIO_DURATION)
assert abs(MAX_AUDIO_DURATION - float(FULL_PREPARE_MAX_DURATION)) < 1e-9
MAX_TARGET_LENGTH = LOCKED_MAX_TARGET_LENGTH
GENERATION_MAX_LENGTH = CFG["generation_max_length"]
NUM_BEAMS = CFG["num_beams"]
SEED = 42

PER_DEVICE_TRAIN_BATCH_SIZE = CFG["per_device_train_batch_size"]
PER_DEVICE_EVAL_BATCH_SIZE = CFG["per_device_eval_batch_size"]
GRADIENT_ACCUMULATION_STEPS = CFG["gradient_accumulation_steps"]
LEARNING_RATE = CFG["learning_rate"]
WARMUP_RATIO = CFG["warmup_ratio"]
WEIGHT_DECAY = CFG["weight_decay"]
NUM_TRAIN_EPOCHS = CFG["num_train_epochs"]
SAVE_STEPS = CFG["save_steps"]
EVAL_STEPS = CFG["eval_steps"]
SAVE_TOTAL_LIMIT = CFG["save_total_limit"]
FP16 = CFG["fp16"]
BF16 = CFG["bf16"]
GRADIENT_CHECKPOINTING = CFG["gradient_checkpointing"]
FREEZE_FEATURE_ENCODER = CFG["freeze_feature_encoder"]
DIRECT_MONITOR_SIZE = CFG["monitor_size"]
METRIC_FOR_BEST_MODEL = LOCKED_METRIC_FOR_BEST_MODEL
GREATER_IS_BETTER = LOCKED_GREATER_IS_BETTER

PILOT_TRAIN_SIZE = 32
PILOT_VALIDATION_SIZE = 8
PILOT_MAX_STEPS = 5

RUNTIME = resolve_direct_runtime_paths(project_root=_PROJECT_ROOT)
ASR_STATE_DIR = Path(os.environ["BAHNAR_DURABLE_ROOT"]) / "bahnar_s2tt" / "full_state" / f"contract_{LOCKED_ASR_PREPARE_CONTRACT_HASH[:16]}"
PATHS = {
    "train_manifest": _PROJECT_ROOT / "data" / "manifests" / "rq1_train.csv",
    "val_manifest": _PROJECT_ROOT / "data" / "manifests" / "rq1_validation.csv",
    "config": _PROJECT_ROOT / DIRECT_CONFIG_RELPATH,
    "artifacts_root": _PROJECT_ROOT / "artifacts" / "notebook05",
    "results_root": _PROJECT_ROOT / "results" / "notebook05",
}
SOURCE_FP = compute_source_fingerprint(_PROJECT_ROOT)
RUNTIME_ENV = assert_locked_direct_runtime()
RUN_ID = str(uuid.uuid4())
PATHS["artifacts"] = PATHS["artifacts_root"] / RUN_ID
PATHS["results"] = PATHS["results_root"] / RUN_ID
PATHS["artifacts"].mkdir(parents=True, exist_ok=True)
PATHS["results"].mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"RUN_ID={RUN_ID} RUN_MODE={RUN_MODE} FULL_STAGE={FULL_STAGE} device={device}")
print("ASR_STATE_DIR=", ASR_STATE_DIR)
print("source_fingerprint=", SOURCE_FP["aggregate_sha256"], "git_head=", SOURCE_FP.get("git_head"))
print(RUNTIME.as_dict())


In [ ]:
# Cell 5 — Pipeline state + shared helpers
pipeline_error = None
pipeline_error_stage = None
STATUS = "RUNNING"
full_status = None
TRACKER = DirectAccessTracker()
OPENED_PATHS = TRACKER.files
LOADED_SPLITS = TRACKER.splits
STATE_DIR = None
direct_contract = None
train_contract = None
train_df = val_df = monitor_df = None
feature_extractor = tokenizer = None
tok_fp = None

def fail_stage(stage, exc):
    global pipeline_error, pipeline_error_stage, STATUS
    if pipeline_error is None:
        pipeline_error = {"exception_type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
        pipeline_error_stage = stage
        STATUS = STATUS_FAILED
    print(f"[STAGE FAILED: {stage}] {type(exc).__name__}: {exc}")

def load_prepared_context():
    global STATE_DIR, direct_contract, train_contract, train_df, val_df, monitor_df, feature_extractor, tokenizer, tok_fp
    ptr = load_latest_prepare_pointer(RUNTIME.durable_state_root)
    STATE_DIR = Path(ptr["state_dir"])
    TRACKER.record_file(STATE_DIR / "direct_prepare_summary.json")
    prep = load_direct_prepare_success(STATE_DIR, expected_contract_hash=ptr["contract_hash"])
    direct_contract = dict(prep["data_contract"])
    assert_direct_data_contract_self_consistent(direct_contract)
    assert_source_fingerprint_matches(prep.get("source_fingerprint"), SOURCE_FP)
    if direct_contract["asr_prepare_contract_hash"] != LOCKED_ASR_PREPARE_CONTRACT_HASH:
        raise RuntimeError("Direct prepared state is not bound to locked NB03 ASR prepare")
    train_df, val_df = load_direct_prepared_frames(STATE_DIR, tracker=TRACKER)
    assert_direct_frames_match_contract(
        train_df, val_df, direct_contract,
        train_path=STATE_DIR / "direct_train_eligible.csv",
        validation_path=STATE_DIR / "direct_validation_eligible.csv",
    )
    feature_extractor, tokenizer = load_direct_processors(
        encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID,
        decoder_revision=DECODER_REVISION, target_lang=TARGET_LANG,
    )
    tok_fp = tokenizer_fingerprint(tokenizer)
    if tok_fp != direct_contract["tokenizer_fingerprint"]:
        raise RuntimeError("mBART tokenizer fingerprint changed vs Direct prepare contract")
    monitor_path = STATE_DIR / "direct_validation_monitor.csv"
    monitor_df = pd.read_csv(monitor_path)
    TRACKER.record_file(monitor_path)
    train_contract = json.loads((STATE_DIR / "direct_training_contract.json").read_text(encoding="utf-8"))
    assert_direct_training_contract_self_consistent(train_contract)
    mon = assert_monitor_matches_validation(monitor_df, val_df, size=DIRECT_MONITOR_SIZE, seed=SEED)
    monitor_file_sha256 = assert_monitor_file_sha256(monitor_path, train_contract["monitor_file_sha256"])
    current_contract = build_direct_training_contract(
        direct_data_contract_hash=direct_contract["contract_hash"], experiment_id=EXPERIMENT_ID,
        encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION,
        target_lang=TARGET_LANG, tokenizer_fingerprint=tok_fp, train_uid_set_hash=direct_contract["train_uid_set_hash"],
        validation_uid_set_hash=direct_contract["validation_uid_set_hash"], monitor_uid_set_hash=mon["uid_set_hash"],
        monitor_pair_hash=mon["pair_hash"], monitor_ordered_row_hash=mon["ordered_row_hash"],
        monitor_file_sha256=monitor_file_sha256, monitor_size=len(monitor_df), seed=SEED, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, fp16=FP16, bf16=BF16,
        gradient_checkpointing=GRADIENT_CHECKPOINTING, save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT, max_target_length=MAX_TARGET_LENGTH, generation_max_length=GENERATION_MAX_LENGTH,
        num_beams=NUM_BEAMS, metric_for_best_model=METRIC_FOR_BEST_MODEL, greater_is_better=GREATER_IS_BETTER,
        freeze_feature_encoder=FREEZE_FEATURE_ENCODER, source_fingerprint_sha256=SOURCE_FP["aggregate_sha256"],
        torch_version=RUNTIME_ENV["torch_version"], transformers_version=RUNTIME_ENV["transformers_version"],
        accelerate_version=RUNTIME_ENV["accelerate_version"],
    )
    assert_direct_training_contracts_match(train_contract, current_contract)
    assert_no_frozen_test_access(TRACKER.files, TRACKER.splits)
    return prep

def hydrate_for_stage(*frames):
    usable = [x for x in frames if x is not None and len(x)]
    return hydrate_direct_audio(
        usable, asr_state_dir=ASR_STATE_DIR, dataset_id=DATASET_ID, parquet_revision=EXPECTED_PARQUET_REVISION,
        audio_cache_dir=RUNTIME.audio_cache_dir, parquet_cache_dir=RUNTIME.hf_parquet_cache_dir,
        sample_rate=SAMPLE_RATE, tracker=TRACKER,
    )

def make_ds(frame):
    return DirectSpeechTranslationDataset(
        frame, feature_extractor=feature_extractor, tokenizer=tokenizer,
        audio_cache_roots=[RUNTIME.audio_cache_dir], sample_rate=SAMPLE_RATE, max_target_length=MAX_TARGET_LENGTH,
    )

def make_args(output_dir, **overrides):
    from transformers import Seq2SeqTrainingArguments
    base = dict(
        output_dir=str(output_dir), per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
        report_to=[], seed=SEED, fp16=FP16 and device=="cuda", bf16=BF16,
        gradient_checkpointing=GRADIENT_CHECKPOINTING, remove_unused_columns=True,
    )
    base.update(overrides)
    return Seq2SeqTrainingArguments(**base)

print("pipeline state initialized")


## Stage A — `prepare`


In [ ]:
# Cell 6 — PREPARE
if FULL_STAGE == "prepare" and pipeline_error is None:
    try:
        matched = load_asr_matched_direct_frames(
            asr_state_dir=ASR_STATE_DIR, train_manifest=PATHS["train_manifest"], validation_manifest=PATHS["val_manifest"],
            expected_asr_contract_hash=LOCKED_ASR_PREPARE_CONTRACT_HASH, tracker=TRACKER,
            expected_train_count=EXPECTED_ASR_TRAIN_COUNT, expected_validation_count=EXPECTED_ASR_VALIDATION_COUNT,
            expected_asr_train_uid_set_hash=CFG["nb03_train_eligible_uid_set_hash"] or None,
            expected_asr_validation_uid_set_hash=CFG["nb03_validation_eligible_uid_set_hash"] or None,
            expected_asr_train_file_sha256=CFG["nb03_train_eligible_file_sha256"] or None,
            expected_asr_validation_file_sha256=CFG["nb03_validation_eligible_file_sha256"] or None,
        )
        train_df, val_df = matched["train"], matched["validation"]
        feature_extractor, tokenizer = load_direct_processors(
            encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID,
            decoder_revision=DECODER_REVISION, target_lang=TARGET_LANG,
        )
        tok_fp = tokenizer_fingerprint(tokenizer)
        target_audit = audit_direct_targets_by_split(
            tokenizer, train_targets=train_df["text_vi_norm"].tolist(), validation_targets=val_df["text_vi_norm"].tolist(),
            max_target_length=MAX_TARGET_LENGTH, max_unk_rate=LOCKED_HARD_MAX_UNK_RATE,
            max_truncation_rate=LOCKED_HARD_MAX_TRUNCATION_RATE,
        )
        if not target_audit["passed"]:
            raise RuntimeError(f"Direct target tokenizer audit failed: {target_audit['failed_checks']}")

        direct_contract = build_direct_data_contract(
            dataset_id=DATASET_ID, dataset_revision=EXPECTED_DATASET_REVISION, parquet_revision=EXPECTED_PARQUET_REVISION,
            asr_prepare_contract_hash=LOCKED_ASR_PREPARE_CONTRACT_HASH, matched_input_policy=MATCHED_INPUT_POLICY,
            train_uid_set_hash=matched["train_uid_set_hash"], validation_uid_set_hash=matched["validation_uid_set_hash"],
            train_ordered_uid_hash=matched["train_ordered_uid_hash"], validation_ordered_uid_hash=matched["validation_ordered_uid_hash"],
            train_pair_hash=matched["train_pair_hash"], validation_pair_hash=matched["validation_pair_hash"],
            train_ordered_row_hash=matched["train_ordered_row_hash"], validation_ordered_row_hash=matched["validation_ordered_row_hash"],
            train_file_sha256=frame_csv_sha256(train_df), validation_file_sha256=frame_csv_sha256(val_df),
            locked_train_manifest_sha256=matched["locked_train_manifest_sha256"],
            locked_validation_manifest_sha256=matched["locked_validation_manifest_sha256"],
            asr_train_eligible_uid_set_hash=matched["asr_train_eligible_uid_set_hash"],
            asr_validation_eligible_uid_set_hash=matched["asr_validation_eligible_uid_set_hash"],
            asr_train_eligible_file_sha256=matched["asr_train_eligible_file_sha256"],
            asr_validation_eligible_file_sha256=matched["asr_validation_eligible_file_sha256"],
            train_count=len(train_df), validation_count=len(val_df), encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION,
            decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION, target_lang=TARGET_LANG,
            tokenizer_fingerprint=tok_fp, target_normalization_version=TARGET_NORMALIZATION_VERSION,
            sample_rate=SAMPLE_RATE, max_audio_duration=MAX_AUDIO_DURATION, max_target_length=MAX_TARGET_LENGTH,
        )
        STATE_DIR = RUNTIME.state_dir(direct_contract["contract_hash"])
        monitor_df = fixed_subset(val_df, DIRECT_MONITOR_SIZE, seed=SEED)
        mon = assert_monitor_matches_validation(monitor_df, val_df, size=DIRECT_MONITOR_SIZE, seed=SEED)
        hparams = build_direct_training_hparams(
            n_train=len(train_df), per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
            num_train_epochs=NUM_TRAIN_EPOCHS, save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT, fp16=FP16, bf16=BF16, gradient_checkpointing=GRADIENT_CHECKPOINTING,
        )
        train_contract = build_direct_training_contract(
            direct_data_contract_hash=direct_contract["contract_hash"], experiment_id=EXPERIMENT_ID,
            encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION,
            target_lang=TARGET_LANG, tokenizer_fingerprint=tok_fp, train_uid_set_hash=matched["train_uid_set_hash"],
            validation_uid_set_hash=matched["validation_uid_set_hash"], monitor_uid_set_hash=mon["uid_set_hash"],
            monitor_pair_hash=mon["pair_hash"], monitor_ordered_row_hash=mon["ordered_row_hash"],
            monitor_file_sha256="0" * 64, monitor_size=len(monitor_df), seed=SEED, learning_rate=LEARNING_RATE,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, num_train_epochs=NUM_TRAIN_EPOCHS,
            warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, fp16=FP16, bf16=BF16,
            gradient_checkpointing=GRADIENT_CHECKPOINTING, save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT, max_target_length=MAX_TARGET_LENGTH, generation_max_length=GENERATION_MAX_LENGTH,
            num_beams=NUM_BEAMS, metric_for_best_model=METRIC_FOR_BEST_MODEL, greater_is_better=GREATER_IS_BETTER,
            freeze_feature_encoder=FREEZE_FEATURE_ENCODER, source_fingerprint_sha256=SOURCE_FP["aggregate_sha256"],
            torch_version=RUNTIME_ENV["torch_version"], transformers_version=RUNTIME_ENV["transformers_version"],
            accelerate_version=RUNTIME_ENV["accelerate_version"],
        )
        summary = {
            "status": STATUS_DIRECT_PREPARE, "contract_hash": direct_contract["contract_hash"],
            "experiment_id": EXPERIMENT_ID, "direct_training_contract_hash": train_contract["direct_training_contract_hash"],
            "data_contract": direct_contract, "train_count": len(train_df), "validation_count": len(val_df),
            "monitor_count": len(monitor_df), "asr_prepare_contract_hash": LOCKED_ASR_PREPARE_CONTRACT_HASH,
            "source_fingerprint": SOURCE_FP, "hparams": hparams,
            "frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
        }
        pointer = persist_direct_prepare(
            STATE_DIR, train=train_df, validation=val_df, summary=summary, target_audit=target_audit,
            training_contract=train_contract, monitor=monitor_df, durable_state_root=RUNTIME.durable_state_root,
        )
        train_contract = pointer["training_contract"]
        summary = pointer["summary"]
        assert_monitor_file_sha256(STATE_DIR / "direct_validation_monitor.csv", train_contract["monitor_file_sha256"])
        assert_direct_frames_match_contract(
            train_df, val_df, direct_contract,
            train_path=STATE_DIR / "direct_train_eligible.csv",
            validation_path=STATE_DIR / "direct_validation_eligible.csv",
        )
        assert_no_frozen_test_access(TRACKER.files, TRACKER.splits)
        full_status = STATUS_DIRECT_PREPARE
        print(json.dumps({k: summary[k] for k in ("status","contract_hash","experiment_id","train_count","validation_count")}, indent=2))
        print("target audit passed:", target_audit["passed"], "val_truncation=", target_audit["validation"]["target"]["truncation_count"])
    except Exception as e:
        fail_stage("direct_prepare", e)


## Stage B — `pilot`


In [ ]:
# Cell 7 — PILOT
if FULL_STAGE == "pilot" and pipeline_error is None:
    try:
        load_prepared_context()
        tr = fixed_subset(train_df, PILOT_TRAIN_SIZE, seed=SEED)
        va = fixed_subset(val_df, PILOT_VALIDATION_SIZE, seed=SEED)
        hydrate_for_stage(tr, va)
        model = load_direct_model(
            encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION,
            tokenizer=tokenizer, target_lang=TARGET_LANG, freeze_feature_encoder=FREEZE_FEATURE_ENCODER,
        )
        if GRADIENT_CHECKPOINTING:
            model.gradient_checkpointing_enable()
        model.to(device)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        from transformers import Seq2SeqTrainer
        args = make_args(
            RUNTIME.local_ckpt_root / PILOT_MARKER / EXPERIMENT_ID, max_steps=PILOT_MAX_STEPS,
            gradient_accumulation_steps=1, save_strategy="no", eval_strategy="no", logging_steps=1,
            predict_with_generate=False, gradient_checkpointing=GRADIENT_CHECKPOINTING,
        )
        trainer = Seq2SeqTrainer(
            model=model, args=args, train_dataset=make_ds(tr), data_collator=DirectDataCollator(feature_extractor, tokenizer),
            processing_class=tokenizer,
        )
        t0=time.time(); result=trainer.train(); runtime=time.time()-t0
        pred_rows = generate_direct_predictions(
            model=model, tokenizer=tokenizer, dataset=make_ds(va), collator=DirectDataCollator(feature_extractor, tokenizer),
            batch_size=1, generation_max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG],
        )
        metrics=evaluate_prediction_rows(pred_rows)
        loss=float(result.metrics.get("train_loss", float("nan")))
        ok=math.isfinite(loss) and len(pred_rows)==len(va) and math.isfinite(float(metrics["sacrebleu"])) and math.isfinite(float(metrics["chrfpp"]))
        payload={
            "status": STATUS_DIRECT_PILOT if ok else STATUS_FAILED, "contract_hash": direct_contract["contract_hash"],
            "experiment_id": EXPERIMENT_ID, "direct_training_contract_hash": train_contract["direct_training_contract_hash"],
            "train_loss": loss, "runtime_sec": runtime, "prediction_count": len(pred_rows),
            "sacrebleu": metrics["sacrebleu"], "chrfpp": metrics["chrfpp"], "parameters": parameter_report(model),
            "gpu_peak_allocated_bytes": int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None,
            "sample_predictions": pred_rows[:5], "frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
        }
        write_json(STATE_DIR / "direct_pilot_summary.json", payload)
        pd.DataFrame(pred_rows).to_csv(PATHS["results"] / "pilot_predictions.csv", index=False)
        if not ok: raise RuntimeError("Direct compatibility pilot failed gates")
        full_status=STATUS_DIRECT_PILOT
        print(json.dumps({k: payload[k] for k in ("status","contract_hash","experiment_id","train_loss")}, indent=2, default=str))
        del trainer, model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        fail_stage("direct_pilot", e)


## Stage C — Resume proof
Run `resume_test_a`, then **restart the kernel/process**, switch to `resume_test_b`. Uses the **same RandomSampler / seed / Trainer class** as full train (not SequentialSampler).


In [ ]:
# Cell 8 — RESUME TEST A / B
if FULL_STAGE in {"resume_test_a","resume_test_b"} and pipeline_error is None:
    try:
        load_prepared_context()
        load_stage_success(
            STATE_DIR / "direct_pilot_summary.json", status=STATUS_DIRECT_PILOT,
            expected_contract_hash=direct_contract["contract_hash"], expected_experiment_id=EXPERIMENT_ID,
            expected_training_contract_hash=train_contract["direct_training_contract_hash"],
        )
        subset = fixed_subset(train_df, RESUME_TEST_SUBSET_SIZE, seed=SEED).reset_index(drop=True)
        rt_dir = RUNTIME.local_ckpt_root / RESUME_TEST_MARKER / EXPERIMENT_ID
        session = new_session_token()
        from transformers import Seq2SeqTrainer
        model = load_direct_model(
            encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION,
            tokenizer=tokenizer, target_lang=TARGET_LANG, freeze_feature_encoder=FREEZE_FEATURE_ENCODER,
        )
        params = parameter_report(model)
        budget_report = assert_direct_checkpoint_budget(n_parameters=params["total_parameters"])
        preflight_direct_local_disk(
            [subset], audio_cache_dir=RUNTIME.audio_cache_dir, parquet_cache_dir=RUNTIME.hf_parquet_cache_dir,
            local_ckpt_dir=rt_dir, n_parameters=params["total_parameters"], save_total_limit=SAVE_TOTAL_LIMIT,
        )
        hydrate_for_stage(subset)
        plan = plan_direct_resume_position(
            subset["record_uid"].astype(str).tolist(), seed=SEED, resume_step=RESUME_TEST_PHASE_A_STEPS,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
        )
        if GRADIENT_CHECKPOINTING: model.gradient_checkpointing_enable()

        if FULL_STAGE == "resume_test_a":
            if (STATE_DIR / "direct_resume_phase_a.json").is_file():
                raise RuntimeError("Phase A artifact already exists. Do not rerun A; restart kernel and run resume_test_b.")
            if rt_dir.exists(): shutil.rmtree(rt_dir)
            rt_dir.mkdir(parents=True, exist_ok=True)
            write_checkpoint_fingerprint(
                rt_dir, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER, global_step=0, extra=train_contract,
            )
            args = make_args(
                rt_dir, max_steps=RESUME_TEST_PHASE_A_STEPS, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                save_strategy="steps", save_steps=RESUME_TEST_PHASE_A_STEPS, save_total_limit=2, eval_strategy="no",
                logging_steps=5, predict_with_generate=False, remove_unused_columns=True,
            )
            trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=make_ds(subset),
                                     data_collator=DirectDataCollator(feature_extractor, tokenizer), processing_class=tokenizer)
            trainer.train()
            ckpt = rt_dir / f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            assert_checkpoint_complete_for_resume(ckpt)
            write_checkpoint_fingerprint(rt_dir, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                                         global_step=RESUME_TEST_PHASE_A_STEPS, extra=train_contract)
            sync_experiment_checkpoints_to_durable(
                rt_dir, STATE_DIR, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER, require_durable=True,
                save_total_limit=2, budget_bytes=budget_report["budget_bytes"],
            )
            payload={
                "status":"PHASE_A_COMPLETE", "contract_hash":direct_contract["contract_hash"],
                "experiment_id":EXPERIMENT_ID, "direct_training_contract_hash":train_contract["direct_training_contract_hash"],
                "training_contract_hash":train_contract["direct_training_contract_hash"], "session":session,
                "global_step":RESUME_TEST_PHASE_A_STEPS, "expected_resume_position":plan,
                "sampler":"accelerate.SeedableRandomSampler", "frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
            }
            write_json(STATE_DIR / "direct_resume_phase_a.json", payload)
            full_status="SUCCESS_DIRECT_RESUME_TEST_PHASE_A"
            print("Phase A durable. NOW RESTART KERNEL, set FULL_STAGE='resume_test_b', and Run All.")
        else:
            phase_a=json.loads((STATE_DIR / "direct_resume_phase_a.json").read_text(encoding="utf-8"))
            if phase_a.get("direct_training_contract_hash") != train_contract["direct_training_contract_hash"]:
                raise RuntimeError("Phase A training contract mismatch")
            if str(phase_a.get("experiment_id")) != EXPERIMENT_ID:
                raise RuntimeError("Phase A experiment_id mismatch")
            cross=assert_cross_session_resume(phase_a, current_session=session)
            restore_experiment_checkpoints_from_durable(
                rt_dir, STATE_DIR, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER, expected_contract=train_contract,
            )
            ckpt=rt_dir / f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            assert_checkpoint_complete_for_resume(ckpt)
            sink={}
            track_collator=make_uid_tracking_collator(DirectDataCollator(feature_extractor, tokenizer), sink)
            TrainerTrack=make_uid_tracking_trainer_cls(make_resume_safe_seq2seq_trainer_cls(), sink)
            proof_cb=make_resume_proof_callback(
                checkpoint_dir=ckpt, expected_resume_step=RESUME_TEST_PHASE_A_STEPS, sink=sink, gradient_accumulation_steps=8,
            )
            args=make_args(
                rt_dir, max_steps=RESUME_TEST_PHASE_B_STEPS, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                save_strategy="no", eval_strategy="no", logging_steps=5, predict_with_generate=False, remove_unused_columns=False,
            )
            trainer=TrainerTrack(model=model,args=args,train_dataset=make_ds(subset),data_collator=track_collator,
                                 processing_class=tokenizer,callbacks=[proof_cb])
            trainer.train(resume_from_checkpoint=str(ckpt))
            proof=summarize_resume_proof(sink, expected_first_uid=phase_a["expected_resume_position"]["expected_first_uid"])
            proof["final_global_step"]=int(trainer.state.global_step)
            verdict=derive_resume_test_status_from_proof(
                proof, phase_a_reached_target=int(phase_a["global_step"])==RESUME_TEST_PHASE_A_STEPS,
                cross_session=cross, used_separate_experiment_dir=(RESUME_TEST_MARKER in str(rt_dir)),
                expected_phase_b_steps=RESUME_TEST_PHASE_B_STEPS,
            )
            payload={
                **verdict, "contract_hash":direct_contract["contract_hash"], "experiment_id":EXPERIMENT_ID,
                "direct_training_contract_hash":train_contract["direct_training_contract_hash"],
                "training_contract_hash":train_contract["direct_training_contract_hash"],
                "frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
            }
            if payload["status"] != "SUCCESS_FULL_RESUME_TEST":
                raise RuntimeError(f"Direct resume proof failed: {payload['failed_checks']}")
            payload["status"]=STATUS_DIRECT_RESUME_TEST
            write_json(STATE_DIR / "direct_resume_test_summary.json", payload)
            full_status=STATUS_DIRECT_RESUME_TEST
            print(json.dumps({k: payload.get(k) for k in ("status","contract_hash","experiment_id")}, indent=2, default=str))
    except Exception as e:
        fail_stage("direct_resume_test", e)


## Stage D — Full training


In [ ]:
# Cell 9 — FULL TRAIN
if FULL_STAGE == "train" and pipeline_error is None:
    try:
        if not ALLOW_FULL_TRAINING:
            raise RuntimeError("Set ALLOW_FULL_TRAINING=True explicitly for FULL_STAGE='train'")
        if device != "cuda": raise RuntimeError("Full Direct training requires CUDA on RunPod")
        load_prepared_context()
        load_stage_success(STATE_DIR / "direct_pilot_summary.json", status=STATUS_DIRECT_PILOT,
            expected_contract_hash=direct_contract["contract_hash"], expected_experiment_id=EXPERIMENT_ID,
            expected_training_contract_hash=train_contract["direct_training_contract_hash"])
        load_stage_success(STATE_DIR / "direct_resume_test_summary.json", status=STATUS_DIRECT_RESUME_TEST,
            expected_contract_hash=direct_contract["contract_hash"], expected_experiment_id=EXPERIMENT_ID,
            expected_training_contract_hash=train_contract["direct_training_contract_hash"])
        exp_dir=RUNTIME.local_ckpt_root / FULL_TRAIN_MARKER / EXPERIMENT_ID
        restore_experiment_checkpoints_from_durable(
            exp_dir, STATE_DIR, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER, expected_contract=train_contract,
        )
        ensure_experiment_fingerprint(
            exp_dir, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER, expected_contract=train_contract, allow_create_if_empty=True,
        )
        resume_ckpt=resolve_resume_checkpoint(
            resume_policy="auto", experiment_dir=exp_dir, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
        )
        print("resume_from_checkpoint:", resume_ckpt)
        model=load_direct_model(
            encoder_id=ENCODER_ID, encoder_revision=ENCODER_REVISION, decoder_id=DECODER_ID, decoder_revision=DECODER_REVISION,
            tokenizer=tokenizer, target_lang=TARGET_LANG, freeze_feature_encoder=FREEZE_FEATURE_ENCODER,
        )
        if GRADIENT_CHECKPOINTING: model.gradient_checkpointing_enable()
        model.generation_config.max_length=GENERATION_MAX_LENGTH
        model.generation_config.num_beams=NUM_BEAMS
        model.to(device)
        params=parameter_report(model)
        budget_report=assert_direct_checkpoint_budget(n_parameters=params["total_parameters"])
        print("checkpoint_budget", budget_report)
        preflight_direct_local_disk(
            [train_df, monitor_df], audio_cache_dir=RUNTIME.audio_cache_dir,
            parquet_cache_dir=RUNTIME.hf_parquet_cache_dir, local_ckpt_dir=exp_dir,
            n_parameters=params["total_parameters"], save_total_limit=SAVE_TOTAL_LIMIT,
        )
        hydrate_for_stage(train_df, monitor_df)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        DirectTrainer = make_resume_safe_seq2seq_trainer_cls()
        sync_cb=make_durable_checkpoint_sync_callback(
            local_experiment_dir=exp_dir, full_state_dir=STATE_DIR, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            training_contract=train_contract, save_total_limit=SAVE_TOTAL_LIMIT,
            budget_bytes=budget_report["budget_bytes"],
        )
        args=make_args(
            exp_dir, num_train_epochs=NUM_TRAIN_EPOCHS, per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            eval_strategy="steps", eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT, load_best_model_at_end=True,
            metric_for_best_model=METRIC_FOR_BEST_MODEL.replace("eval_", ""), greater_is_better=GREATER_IS_BETTER,
            predict_with_generate=True, generation_max_length=GENERATION_MAX_LENGTH, generation_num_beams=NUM_BEAMS,
            logging_steps=100, remove_unused_columns=True,
        )
        def compute_metrics(eval_preds):
            return compute_direct_seq2seq_metrics(eval_preds, tokenizer=tokenizer)
        trainer=DirectTrainer(
            model=model,args=args,train_dataset=make_ds(train_df),eval_dataset=make_ds(monitor_df),
            data_collator=DirectDataCollator(feature_extractor,tokenizer),processing_class=tokenizer,
            compute_metrics=compute_metrics,callbacks=[sync_cb],
        )
        t0=time.time(); result=trainer.train(resume_from_checkpoint=resume_ckpt); train_runtime=time.time()-t0
        metrics=trainer.evaluate()
        write_checkpoint_fingerprint(exp_dir, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
                                     global_step=int(trainer.state.global_step), extra=train_contract, overwrite=True)
        best_name=Path(str(trainer.state.best_model_checkpoint or "")).name
        validated_best=validate_direct_best_checkpoint(
            experiment_dir=exp_dir, experiment_id=EXPERIMENT_ID,
            best_checkpoint=best_name, expected_contract=train_contract,
        )
        sync_experiment_checkpoints_to_durable(
            exp_dir, STATE_DIR, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER, require_durable=True,
            save_total_limit=SAVE_TOTAL_LIMIT, budget_bytes=budget_report["budget_bytes"],
            best_checkpoint_name=validated_best["best_checkpoint_name"],
        )
        steps=list_step_checkpoints(exp_dir)
        latest=steps[-1] if steps else exp_dir
        max_steps=int(getattr(trainer.state, "max_steps", 0) or 0)
        durable_ok=False
        durable_best=None
        try:
            durable_best=restore_direct_best_checkpoint(
                state_dir=STATE_DIR, experiment_id=EXPERIMENT_ID,
                train_summary={"best_checkpoint":best_name,"best_checkpoint_name":best_name},
                local_experiment_dir=exp_dir, expected_contract=train_contract,
            )
            durable_ok=Path(durable_best).name == best_name
            if not durable_ok:
                print("durable LATEST did not restore best checkpoint:", durable_best, best_name)
        except Exception as durable_exc:
            print("durable best resolve failed:", durable_exc)
        best_path=Path(durable_best) if durable_ok else Path(validated_best["best_checkpoint_path"])
        verdict=derive_direct_training_status(
            global_step=int(trainer.state.global_step), max_steps=max_steps, metrics=metrics,
            latest_checkpoint=latest, best_checkpoint=best_path, durable_best_resolved=durable_ok,
            frozen_test_accessed=frozen_flag_from_tracker(TRACKER),
        )
        status=verdict["status"]
        payload={
            "status":status,"contract_hash":direct_contract["contract_hash"],
            "experiment_id":EXPERIMENT_ID,"direct_training_contract_hash":train_contract["direct_training_contract_hash"],
            "global_step":int(trainer.state.global_step),"max_steps":max_steps,
            "best_checkpoint_name":best_name,"best_checkpoint":best_name,"latest_checkpoint_name":latest.name,
            "metrics_monitor":metrics,"metrics_note":"eval_* is fixed 256-sample validation monitor; full validation is FULL_STAGE=evaluate",
            "train_count":len(train_df),"validation_count":len(val_df),"monitor_count":len(monitor_df),
            "parameters":params,"train_runtime_sec":train_runtime,
            "gpu_peak_allocated_bytes":int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None,
            "checkpoint_budget":budget_report,"frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
            "success_checks":verdict["checks"],"failed_checks":verdict["failed_checks"],
        }
        write_json(STATE_DIR / "direct_train_summary.json",payload)
        write_json(PATHS["results"] / "direct_train_summary.json",payload)
        if status != STATUS_DIRECT_TRAINING:
            raise RuntimeError(f"Direct full training failed success gates: {verdict['failed_checks']}")
        full_status=status
        print(json.dumps({k:payload[k] for k in ("status","contract_hash","experiment_id","best_checkpoint_name")},indent=2,default=str))
    except Exception as e:
        fail_stage("direct_train",e)


## Stage E — Full validation evaluation


In [ ]:
# Cell 10 — EVALUATE
if FULL_STAGE == "evaluate" and pipeline_error is None:
    try:
        load_prepared_context()
        train_summary=load_stage_success(STATE_DIR / "direct_train_summary.json",status=STATUS_DIRECT_TRAINING,
            expected_contract_hash=direct_contract["contract_hash"], expected_experiment_id=EXPERIMENT_ID,
            expected_training_contract_hash=train_contract["direct_training_contract_hash"])
        hydrate_for_stage(val_df)
        exp_dir=RUNTIME.local_ckpt_root / FULL_TRAIN_MARKER / EXPERIMENT_ID
        best_dir=restore_direct_best_checkpoint(
            state_dir=STATE_DIR, experiment_id=EXPERIMENT_ID, train_summary=train_summary,
            local_experiment_dir=exp_dir, expected_contract=train_contract,
        )
        from transformers import SpeechEncoderDecoderModel
        model=SpeechEncoderDecoderModel.from_pretrained(str(best_dir))
        forced_bos=int(tokenizer.lang_code_to_id[TARGET_LANG])
        model.generation_config.forced_bos_token_id=forced_bos
        model.generation_config.max_length=GENERATION_MAX_LENGTH
        model.generation_config.num_beams=NUM_BEAMS
        model.to(device)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        t0=time.time()
        rows=generate_direct_predictions(
            model=model,tokenizer=tokenizer,dataset=make_ds(val_df),collator=DirectDataCollator(feature_extractor,tokenizer),
            batch_size=PER_DEVICE_EVAL_BATCH_SIZE,generation_max_length=GENERATION_MAX_LENGTH,num_beams=NUM_BEAMS,
            forced_bos_token_id=forced_bos,
        )
        runtime=time.time()-t0
        metrics=evaluate_prediction_rows(rows)
        finite=math.isfinite(float(metrics["sacrebleu"])) and math.isfinite(float(metrics["chrfpp"]))
        status=STATUS_DIRECT_EVALUATE if len(rows)==len(val_df) and finite else STATUS_FAILED
        gen_cfg=build_direct_generation_config(
            target_lang=TARGET_LANG,forced_bos_token_id=forced_bos,decoder_start_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id,
            generation_max_length=GENERATION_MAX_LENGTH,num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL,greater_is_better=GREATER_IS_BETTER,
        )
        payload={
            "status":status,"contract_hash":direct_contract["contract_hash"],
            "direct_training_contract_hash":train_contract["direct_training_contract_hash"],
            "experiment_id":EXPERIMENT_ID,"best_checkpoint_name":Path(best_dir).name,"best_checkpoint_durable":str(best_dir),
            "n_validation":len(val_df),"prediction_count":len(rows),"sacrebleu":metrics["sacrebleu"],
            "chrfpp":metrics["chrfpp"],"sacrebleu_version":metrics.get("sacrebleu_version"),
            "sacrebleu_signature":metrics.get("sacrebleu_signature"),"chrf_signature":metrics.get("chrf_signature"),
            "generation_config":gen_cfg,"evaluation_runtime_sec":runtime,
            "samples_per_second":float(len(rows)/runtime) if runtime>0 else None,
            "gpu_peak_allocated_bytes":int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None,
            "parameters":parameter_report(model),"metrics_scope":"full_matched_validation",
            "reference_source":"text_vi_norm_by_uid","frozen_test_accessed": frozen_flag_from_tracker(TRACKER),
        }
        pd.DataFrame(rows).to_csv(PATHS["results"] / "direct_validation_predictions.csv",index=False)
        write_json(STATE_DIR / "direct_evaluate_summary.json",payload)
        write_json(PATHS["results"] / "direct_evaluate_summary.json",payload)
        assert_no_frozen_test_access(TRACKER.files, TRACKER.splits)
        if status != STATUS_DIRECT_EVALUATE: raise RuntimeError("Direct full validation evaluate failed gates")
        full_status=status
        print(json.dumps({k:payload[k] for k in ("status","contract_hash","n_validation","sacrebleu")},indent=2,default=str))
    except Exception as e:
        fail_stage("direct_evaluate",e)


In [ ]:
# Cell 11 — Final status / handoff
if pipeline_error is not None:
    STATUS=STATUS_FAILED
elif full_status:
    STATUS=full_status
else:
    STATUS=STATUS_FAILED

assert_no_frozen_test_access(TRACKER.files, TRACKER.splits)
hand=derive_direct_handoff(STATUS, frozen_test_accessed=frozen_flag_from_tracker(TRACKER))
run_summary={
    "run_id":RUN_ID,"status":STATUS,"run_mode":RUN_MODE,"full_stage":FULL_STAGE,"experiment_id":EXPERIMENT_ID,
    "encoder_id":ENCODER_ID,"encoder_revision":ENCODER_REVISION,"decoder_id":DECODER_ID,"decoder_revision":DECODER_REVISION,
    "target_lang":TARGET_LANG,"contract_hash":(direct_contract or {}).get("contract_hash") if direct_contract else None,
    "direct_training_contract_hash": (train_contract or {}).get("direct_training_contract_hash") if train_contract else None,
    "source_fingerprint": SOURCE_FP, "pipeline_error_stage":pipeline_error_stage,"pipeline_error":pipeline_error,
    "opened_paths":list(TRACKER.files),"loaded_splits":list(TRACKER.splits),"access_tracker":TRACKER.as_dict(),
    **hand,
}
write_json(PATHS["results"] / "run_summary.json",run_summary)
print("="*64)
print("NOTEBOOK 05 STATUS:",STATUS)
print("FULL_STAGE:",FULL_STAGE)
print("READY FOR RQ1 EVALUATION:",hand["ready_for_rq1_evaluation"])
print("READY FOR RQ1 FINAL:",hand["ready_for_rq1_final"])
print("FROZEN TEST ACCESSED:",hand["frozen_test_accessed"])
print("="*64)
if STATUS==STATUS_FAILED:
    raise RuntimeError(f"NOTEBOOK 05 FAILED stage={pipeline_error_stage} reason={pipeline_error}")


In [ ]:
# Cell 12 — Durable export
def _ver(name):
    try:
        m=__import__(name); return getattr(m,"__version__",None)
    except Exception: return None
_env={
    "python":platform.python_version(),"platform":platform.platform(),"torch":getattr(torch,"__version__",None),
    "cuda_available":bool(torch.cuda.is_available()),"gpu_name":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers":_ver("transformers"),"sacrebleu":_ver("sacrebleu"),"sentencepiece":_ver("sentencepiece"),
    "BAHNAR_LOCAL_ROOT":os.environ.get("BAHNAR_LOCAL_ROOT"),"BAHNAR_DURABLE_ROOT":os.environ.get("BAHNAR_DURABLE_ROOT"),
    "BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES":os.environ.get("BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES"),
    "source_fingerprint": SOURCE_FP,
}
write_json(PATHS["results"] / "environment.json",_env)
if STATE_DIR is not None:
    export_dir=export_notebook05_run(
        export_root=RUNTIME.export_root,run_id=RUN_ID,artifacts_dir=PATHS["artifacts"],results_dir=PATHS["results"],
        state_dir=STATE_DIR,pointer={"run_id":RUN_ID,"status":STATUS,"full_stage":FULL_STAGE,"state_dir":str(STATE_DIR),
                                 "experiment_id":EXPERIMENT_ID,"contract_hash":(direct_contract or {}).get("contract_hash") if direct_contract else None},
    )
    print("Exported:",export_dir)
print("Environment:",json.dumps({k:_env[k] for k in ("python","torch","BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES")},indent=2,default=str))
